# Neural Amp Modeler ("Easy Mode" Trainer)
**Note**:
This notebook is meant to be used on [Google Colab](https://colab.research.google.com/github/sdatkinson/neural-amp-modeler/blob/main/bin/train/easy_colab.ipynb).

🔶**Before you run**🔶

Make sure to get a GPU! (From the upper-left menu, click Runtime->Change runtime type->Select "GPU" from the "Hardware accelerator dropdown menu)

## Step 1: Get data
* **Download the reamp signal.** Here: [v3_0_0.wav](https://drive.google.com/file/d/1Pgf8PdE0rKB1TD4TRPKbpNo1ByR3IOm9/view?usp=drive_link).
* **Reamp your gear.** Then reamp the gear you want to model using it. Save that reamp as "output.wav". *Note: Use 48kHz, 24-bit, mono.* For other sample rates, use [the CLI trainer](https://github.com/sdatkinson/neural-amp-modeler).
* **Upload your files.** Upload the input (DI) and output (amped) files you want to use by clicking the Folder icon on the left ⬅ and then clicking the upload icon or by dragging the files into the panel.

## Step 2: Train!
Configure your training run below, then hit the Play button to start training!

🕙NOTE: At default settings, training will take about 10 minutes.🕙

In [ ]:
from pathlib import Path

# If your app already places input.wav / output.wav into this runtime
# (e.g. by mounting Google Drive and copying them here, or via the Colab
# API before this cell runs), this cell is a no-op. Otherwise it falls
# back to a manual upload prompt so the notebook still works standalone.
required = ["input.wav", "output.wav"]
missing = [f for f in required if not Path(f).exists()]

if missing:
    print(f"Missing {missing} in the runtime — please upload them now.")
    from google.colab import files
    uploaded = files.upload()
    still_missing = [f for f in required if not Path(f).exists()]
    if still_missing:
        raise FileNotFoundError(
            f"Still missing {still_missing} after upload. Make sure the files "
            "are named exactly 'input.wav' and 'output.wav'."
        )
else:
    print("Found input.wav and output.wav — ready to train.")


In [ ]:
try:
    import nam
except ImportError as e:
    print("Installing NAM into Colab. This should take under 2 minutes.")
    # Check what we're starting with (Issue 399)
    !if [ ! -d logs ]; then mkdir logs; fi
    !pip list > logs/packages.log
    !pip install neural-amp-modeler > logs/install.log
    # Hint: use the next line instead for the very latest!
    # !pip install git+https://github.com/sdatkinson/neural-amp-modeler.git@main

from nam.train.colab import run
from nam.models.metadata import GearType, ToneType, UserMetadata

%load_ext tensorboard

import json
import re
import sys
import time
import traceback
from functools import partial
from pathlib import Path

import ipywidgets as widgets

# =====================================================================
# Training parameters
# =====================================================================
# LOCKED — not exposed to the end user in the app. The web UI's "Start
# Capture" step always trains with this fixed epoch count; the value is
# only ever changed here in the notebook, not from the frontend.
EPOCHS = 20  # TODO: bump to 600 for production runs (kept low for testing)

architecture = "lite"  # ["standard", "lite", "feather", "nano"]
latency_samples = "auto"
fit_cab = False
ignore_checks = False

# Metadata
use_metadata = False
name = "My model"
modeled_by = "Your name"
gear_make = "GearCo"
gear_model = "GearName"
gear_type = "amp"      # ["amp", "pedal", "pedal_amp", "amp_cab", "amp_pedal_cab", "preamp", "studio"]
tone_type = "clean"    # ["clean", "overdrive", "crunch", "hi_gain", "fuzz"]

# =====================================================================
# Live progress reporting
# =====================================================================
# Every state change is written to PROGRESS_PATH as JSON, and every log
# line (real NAM/Lightning output, not a simulation) is appended to
# LOG_PATH. The app polls these two files to drive the "Cloud Training"
# step's progress bar / percentage / log box with real data.
#
# If this Colab is set up to write into a mounted Google Drive folder
# instead of the local /content runtime (so the desktop app can read it
# without a live connection to this kernel), just change PROGRESS_PATH /
# LOG_PATH to point at that Drive folder, e.g.:
#   PROGRESS_PATH = Path("/content/drive/MyDrive/Namplifier/<capture_id>/progress.json")
PROGRESS_PATH = Path("progress.json")
LOG_PATH = Path("training.log")

_EPOCH_RE = re.compile(r"Epoch\s+(\d+):\s+(\d+)%")


def _now():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def write_progress(status, percent=None, epoch=None, total_epochs=EPOCHS, message=""):
    payload = {
        "status": status,          # "queued" | "training" | "done" | "error"
        "percent": percent,
        "epoch": epoch,
        "total_epochs": total_epochs,
        "message": message,
        "updated_at": _now(),
    }
    PROGRESS_PATH.write_text(json.dumps(payload))


def log_line(text):
    line = f"[{_now()}] {text}"
    with LOG_PATH.open("a") as f:
        f.write(line + "\n")


class _TeeStream:
    """Mirrors stdout/stderr to the real console AND to training.log,
    parsing Lightning's epoch progress bar to update progress.json."""

    def __init__(self, real_stream):
        self.real_stream = real_stream
        self._buf = ""

    def write(self, data):
        self.real_stream.write(data)
        self._buf += data
        while "\n" in self._buf or "\r" in self._buf:
            sep = "\n" if "\n" in self._buf else "\r"
            line, self._buf = self._buf.split(sep, 1)
            line = line.strip()
            if not line:
                continue
            log_line(line)
            m = _EPOCH_RE.search(line)
            if m:
                epoch, pct_in_epoch = int(m.group(1)), int(m.group(2))
                overall_pct = min(
                    99, round(((epoch + pct_in_epoch / 100) / EPOCHS) * 100)
                )
                write_progress(
                    "training",
                    percent=overall_pct,
                    epoch=epoch,
                    message=line,
                )

    def flush(self):
        self.real_stream.flush()


def _verbose_enum(E, val):
    try:
        return E(val)
    except ValueError as e:
        raise ValueError(
            str(e)
            + "\nValid choices are: "
            + ", ".join(list(x.value for x in E))
        )


def _parse_latency(ls: str):
    if ls.lower() == "auto":
        return None
    try:
        return int(ls)
    except ValueError as e:
        raise ValueError(
            f"Invalid value for latency {ls} was given. Either use 'auto' or provide "
            f"the reamp latency, in samples.\nOriginal error:\n\n{e}"
        )


user_metadata = None if not use_metadata else UserMetadata(
    name=name,
    modeled_by=modeled_by,
    gear_make=gear_make,
    gear_model=gear_model,
    gear_type=_verbose_enum(GearType, gear_type.lower()),
    tone_type=_verbose_enum(ToneType, tone_type.lower())
)
run_partial = partial(run, user_metadata=user_metadata)

%tensorboard --logdir /content/lightning_logs

LOG_PATH.write_text("")  # fresh log for this run
write_progress("queued", percent=0, epoch=0, message="Eğitim kuyruğuna alındı.")

_stdout, _stderr = sys.stdout, sys.stderr
sys.stdout, sys.stderr = _TeeStream(_stdout), _TeeStream(_stderr)
try:
    write_progress("training", percent=1, epoch=0, message="Eğitim başladı.")
    run(
        epochs=EPOCHS,
        delay=_parse_latency(latency_samples),
        user_metadata=user_metadata,
        ignore_checks=ignore_checks,
    )
    write_progress("done", percent=100, epoch=EPOCHS, message="Model eğitimi tamamlandı, .nam dosyası dışa aktarıldı.")
except Exception as e:
    write_progress("error", message=f"{type(e).__name__}: {e}")
    log_line("ERROR: " + traceback.format_exc())
    raise
finally:
    sys.stdout, sys.stderr = _stdout, _stderr


## Step 3: Check the results and download your model
We're done!

Have a look at the plot above to see how your model compares to the real gear you're modeling.
Hopefully it looks good!
Go to the file browser on the left panel ⬅ and download `model.nam` from the `exported_model` directory (you may need to hit the refresh button).

Additionally, if you want to continue to train this model later you can download the lightning model artifacts from `lightning_logs`. If not, that's fine too.

# 🎸 **ENJOY!** 🎸